# 01 -- Archetype Physics

**Purpose (PROJECT.md Section 8.7):** geometry, three envelope states, H/C/tau derivation and hand-check. Week 1.

Full archetype specification and sourcing: `PROJECT.md` Section 8.3.
Every numeric input, with its evidentiary status (GROUNDED / PROVISIONAL / DELIBERATE / FORWARD-LOOKING / OMITTED): `configs/tenure_insulation_assumptions.yml`.

**No-double-counting reminder (PROJECT.md Section 2.3):** infiltration/reveal-leakage effects belong in the ACH term, not folded into the window U-value. Keep physical mechanisms in their own lane.


In [1]:
import sys

# PROJECT.md Section 4.1: first cell of every notebook must verify the
# active Python executable path, to catch cross-project kernel mix-ups
# before they silently corrupt a run.
print("Python executable:", sys.executable)
assert "thermal-counterfactual-gb" in sys.executable, (
    "Wrong kernel selected -- pick the 'thermal-counterfactual-gb' kernel, "
    "not a default/global one. Run setup.sh first if it doesn't exist yet."
)

import numpy as np
import polars as pl
import scipy
import matplotlib
print("polars:", pl.__version__, "| numpy:", np.__version__, "| scipy:", scipy.__version__)


Python executable: /tmp/kernelenv/thermal-counterfactual-gb/bin/python3


polars: 1.43.2 | numpy: 2.2.6 | scipy: 1.15.3


## 1. Load assumptions

In [2]:
import yaml
from pathlib import Path

CONFIG_PATH = Path("../configs/tenure_insulation_assumptions.yml")
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

print(f"Loaded {len(cfg)} top-level assumption groups from {CONFIG_PATH}")


Loaded 18 top-level assumption groups from ../configs/tenure_insulation_assumptions.yml


## 2. Build the three envelope states

Physics functions live in `src/thermal_counterfactual_gb/physics.py`, not here --
notebooks import shared logic instead of duplicating it (PROJECT.md Section 5,
"Small Functions, Clear Contracts"). If the package isn't picking up via
`uv sync`, the `sys.path` fallback below still works.


In [3]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../src").resolve()))  # fallback if not installed editable

from thermal_counterfactual_gb.physics import (
    EnvelopeState,
    heat_loss_coefficient_w_per_k,
    thermal_time_constant_hours,
    coastdown_hours,
    peak_electrical_demand_kw,
)

geometry = cfg["geometry"]
envelope_cfg = cfg["envelope_states"]
c_kwh_per_k = cfg["thermal_capacity"]["c_kwh_per_k"]
comfort = cfg["comfort_band"]
cop = cfg["heat_pump"]["cop_at_cold_snap"]
t_outdoor_c = cfg["cold_snap_event"]["design_outdoor_temp_c"]
internal_gains_w = cfg["internal_gains_w"]["point"]  # external review: house is not an empty box


def build_state(name):
    s = envelope_cfg[name]
    return EnvelopeState(
        wall_u_w_per_m2k=s["wall_u_w_per_m2k"],
        roof_u_w_per_m2k=s["roof_u_w_per_m2k"],
        floor_u_w_per_m2k=s["floor_u_w_per_m2k"],
        window_u_w_per_m2k=s["window_u_w_per_m2k"],
        infiltration_ach=s["infiltration_ach"],
    )


states = {name: build_state(name) for name in ["baseline", "swi_only", "epc_c_package"]}
states


{'baseline': EnvelopeState(wall_u_w_per_m2k=1.7, roof_u_w_per_m2k=2.3, floor_u_w_per_m2k=1.5, window_u_w_per_m2k=3.1, infiltration_ach=1.0),
 'swi_only': EnvelopeState(wall_u_w_per_m2k=0.3, roof_u_w_per_m2k=2.3, floor_u_w_per_m2k=1.5, window_u_w_per_m2k=3.1, infiltration_ach=1.0),
 'epc_c_package': EnvelopeState(wall_u_w_per_m2k=0.3, roof_u_w_per_m2k=0.16, floor_u_w_per_m2k=0.25, window_u_w_per_m2k=1.6, infiltration_ach=0.8)}

## 3. Heat loss coefficient (H), time constant (tau), coastdown, and peak demand

H = sum(area_i * U_i) + 0.33 * ACH * heated_volume_m3
tau = C / H
coastdown: Newton's-law-of-cooling exponential decay from preheat ceiling to comfort minimum
peak demand: steady-state H * delta_T / COP, delta_T from normal setpoint to design outdoor temperature


In [4]:
results = {}
for name, state in states.items():
    h = heat_loss_coefficient_w_per_k(
        state,
        wall_area_m2=geometry["opaque_wall_area_m2"],
        roof_area_m2=geometry["roof_area_m2"],
        floor_area_m2=geometry["ground_floor_area_m2"],
        window_area_m2=geometry["window_area_m2"],
        heated_volume_m3=geometry["heated_volume_m3"],
    )
    tau = thermal_time_constant_hours(c_kwh_per_k, h)
    coastdown = coastdown_hours(
        tau_hours=tau,
        t_start_c=comfort["preheat_ceiling_c"],
        t_min_c=comfort["minimum_c"],
        t_outdoor_c=t_outdoor_c,
        h_w_per_k=h,
        internal_gains_w=internal_gains_w,
    )
    delta_t_peak = comfort["normal_setpoint_c"] - t_outdoor_c
    peak_kw = peak_electrical_demand_kw(h, delta_t_peak, cop, internal_gains_w=internal_gains_w)
    results[name] = {
        "h_w_per_k": h,
        "tau_hours": tau,
        "coastdown_hours": coastdown,
        "peak_kw": peak_kw,
    }

for name, r in results.items():
    h_val = r["h_w_per_k"]
    tau_val = r["tau_hours"]
    coast_val = r["coastdown_hours"]
    peak_val = r["peak_kw"]
    print(f"{name:15s}  H={h_val:7.2f} W/K   tau={tau_val:6.2f} h   coastdown={coast_val:5.2f} h   peak={peak_val:5.3f} kW")


baseline         H= 272.75 W/K   tau= 36.66 h   coastdown= 5.00 h   peak=2.458 kW
swi_only         H= 230.75 W/K   tau= 43.34 h   coastdown= 5.98 h   peak=2.055 kW
epc_c_package    H=  85.55 W/K   tau=116.89 h   coastdown=18.67 h   peak=0.661 kW


## 4. Hand-check against PROJECT.md Section 8.3

Tolerances are deliberately a bit looser than "exact" because the reference
values in `configs/tenure_insulation_assumptions.yml -> resolved_physics` are
hand-rounded for readability, not stored at full precision. This still fails
loudly (PROJECT.md Section 5.3) on anything that would matter -- a wrong
formula, a unit error, a mixed-up state -- while tolerating rounding noise.
If a reviewer wants full precision, the recomputed values a few cells above
are the ground truth, not the config file's rounded copies of them.


In [5]:
expected = cfg["resolved_physics"]
name_map = {"baseline": "baseline", "swi_only": "swi_only", "epc_c_package": "epc_c"}
tolerances = {"h_w_per_k": 1.0, "tau_hours": 1.2, "coastdown_hours": 0.15, "peak_kw": 0.02}
expected_key = {
    "h_w_per_k": "h_w_per_k",
    "tau_hours": "tau_hours",
    "coastdown_hours": "coastdown_hours_at_minus3c",
    "peak_kw": "peak_kw_at_cop2_5",
}

for computed_name, expected_name in name_map.items():
    exp = expected[expected_name]
    got = results[computed_name]
    for field, tol in tolerances.items():
        computed_val = got[field]
        expected_val = exp[expected_key[field]]
        diff = abs(computed_val - expected_val)
        assert diff <= tol, f"{computed_name}.{field} mismatch: computed {computed_val:.3f}, expected ~{expected_val}, diff {diff:.3f} > tolerance {tol}"

print("All three states hand-check against PROJECT.md Section 8.3 resolved values.")


All three states hand-check against PROJECT.md Section 8.3 resolved values.


## 5. Headline: baseline to EPC-C peak reduction

In [6]:
baseline_peak = results["baseline"]["peak_kw"]
epc_c_peak = results["epc_c_package"]["peak_kw"]
peak_reduction_kw_per_home = baseline_peak - epc_c_peak
peak_reduction_mw_per_1000_homes = peak_reduction_kw_per_home  # kW/home * 1000 homes / 1000 (W->MW) = same number

print(f"Baseline -> EPC-C peak reduction: {peak_reduction_kw_per_home:.2f} kW/home = {peak_reduction_mw_per_1000_homes:.2f} MW per 1,000 homes")

baseline_coast = results["baseline"]["coastdown_hours"]
epc_c_coast = results["epc_c_package"]["coastdown_hours"]

headline = (
    f"Fabric retrofit from baseline to an EPC-C package cuts peak electrical heating demand by "
    f"{peak_reduction_kw_per_home:.2f} kW per pre-1919 solid-wall mid-terrace home, because extending "
    f"the coastdown window from {baseline_coast:.1f}h to {epc_c_coast:.1f}h lets a heat pump stay off "
    f"through the evening peak. This covers space-heating demand only (DHW and summer overheating are "
    f"omitted), assumes a conservative COP of 2.5, and now nets off a {internal_gains_w:.0f}W constant "
    f"internal gain (occupants/appliances/cooking -- external review point, PROVISIONAL, see config) "
    f"from every state equally, but still ignores solar gains (far more time- and orientation-variable, "
    f"handled separately for annual economics only in Notebook 05). The kW reduction figure is a "
    f"steady-state result independent of the 1R1C-vs-2R2C modelling choice below -- only the "
    f"coastdown-hours figures carry that risk; Section 7 quantifies it, not just flags it."
)
print()
print("Modelling Prose form (PROJECT.md Section 8.3):")
print(headline)


Baseline -> EPC-C peak reduction: 1.80 kW/home = 1.80 MW per 1,000 homes

Modelling Prose form (PROJECT.md Section 8.3):
Fabric retrofit from baseline to an EPC-C package cuts peak electrical heating demand by 1.80 kW per pre-1919 solid-wall mid-terrace home, because extending the coastdown window from 5.0h to 18.7h lets a heat pump stay off through the evening peak. This covers space-heating demand only (DHW and summer overheating are omitted), assumes a conservative COP of 2.5, and now nets off a 400W constant internal gain (occupants/appliances/cooking -- external review point, PROVISIONAL, see config) from every state equally, but still ignores solar gains (far more time- and orientation-variable, handled separately for annual economics only in Notebook 05). The kW reduction figure is a steady-state result independent of the 1R1C-vs-2R2C modelling choice below -- only the coastdown-hours figures carry that risk; Section 7 quantifies it, not just flags it.


## 6. Heat-loss coefficient breakdown (SAP-style transparency check)

External review asked to see H as an explicit wall/roof/floor/window/infiltration breakdown rather than a single summed number, to make it directly checkable against a reviewer's own SAP-style estimate.

In [7]:
print("Heat-loss coefficient breakdown by term, W/K (SAP-style transparency check):")
print(f"{'state':>14} {'wall':>8} {'roof':>8} {'floor':>8} {'window':>8} {'infilt.':>9} {'TOTAL':>8}")
for name, state in states.items():
    wall_term = geometry["opaque_wall_area_m2"] * state.wall_u_w_per_m2k
    roof_term = geometry["roof_area_m2"] * state.roof_u_w_per_m2k
    floor_term = geometry["ground_floor_area_m2"] * state.floor_u_w_per_m2k
    window_term = geometry["window_area_m2"] * state.window_u_w_per_m2k
    infiltration_term = 0.33 * state.infiltration_ach * geometry["heated_volume_m3"]
    total_term = wall_term + roof_term + floor_term + window_term + infiltration_term
    print(f"{name:>14} {wall_term:>8.1f} {roof_term:>8.1f} {floor_term:>8.1f} {window_term:>8.1f} {infiltration_term:>9.1f} {total_term:>8.1f}")

print()
print(
    "External review flagged that the baseline H might look low against a naive SAP-style guess for "
    "unretrofitted pre-1919 stock (e.g. one assuming the OLD, superseded 2.1 W/m2K wall default and "
    "0.7+ ACH). This project's baseline instead uses the RESOLVED, GROUNDED RdSAP v9.93 default (1.7, "
    "not 2.1 -- independently verified against BRE's 2014 in-situ measurements, PROJECT.md Section 8.3) "
    "and a 1.0 ACH infiltration rate. The breakdown above is the full working, term by term -- a "
    "reviewer can check it against their own SAP-style estimate directly instead of trusting one "
    "summed H number."
)
print()
print(
    f"Sanity check against external review's suggested range: this project's baseline H = "
    f"{results['baseline']['h_w_per_k']:.0f} W/K falls within the 250-300 W/K range independently "
    f"proposed for a 'truly leaky' unretrofitted pre-1919 mid-terrace -- no correction needed here; "
    f"the earlier concern (an artificially low ~120 W/K baseline) does not describe this model."
)
print()
print(
    f"Second external review point, DOES change this notebook's numbers: {internal_gains_w:.0f}W of "
    f"constant internal gains (occupants, appliances, cooking -- config: internal_gains_w, PROVISIONAL) "
    f"are now netted off every state's heating demand equally. Against baseline's "
    f"{results['baseline']['h_w_per_k']:.0f} W/K loss, that is a small fraction of the load; against "
    f"EPC-C's {results['epc_c_package']['h_w_per_k']:.0f} W/K, it is proportionally much larger -- see "
    f"the peak-kW and coastdown figures above, both computed WITH this gain already applied."
)


Heat-loss coefficient breakdown by term, W/K (SAP-style transparency check):
         state     wall     roof    floor   window   infilt.    TOTAL
      baseline     51.0     80.5     52.5     31.0      57.8    272.8
      swi_only      9.0     80.5     52.5     31.0      57.8    230.8
 epc_c_package      9.0      5.6      8.8     16.0      46.2     85.6

External review flagged that the baseline H might look low against a naive SAP-style guess for unretrofitted pre-1919 stock (e.g. one assuming the OLD, superseded 2.1 W/m2K wall default and 0.7+ ACH). This project's baseline instead uses the RESOLVED, GROUNDED RdSAP v9.93 default (1.7, not 2.1 -- independently verified against BRE's 2014 in-situ measurements, PROJECT.md Section 8.3) and a 1.0 ACH infiltration rate. The breakdown above is the full working, term by term -- a reviewer can check it against their own SAP-style estimate directly instead of trusting one summed H number.

Sanity check against external review's suggested range

## 7. Structural sensitivity: does the 1R1C coastdown survive a 2-node model?

**External review flagged this as the single biggest physical risk to the project's headline claim.** A single-node (1R1C) model has no separate, small-capacity "air" node, so it cannot show indoor air cooling faster than the whole-building average temperature implies. Since what a thermostat (and an occupant) actually responds to is air temperature, not a lumped average, this matters most in exactly the first few hours after heating stops -- which is exactly the 4-hour peak window this project's headline claims are about.

This section builds a physically-grounded 2-node (air / mass) RC network -- see `two_node_conductances_w_per_k()` in `src/thermal_counterfactual_gb/physics.py` for the full derivation, including the no-double-counting validation against the 1R1C model -- and sweeps the one genuinely ungrounded parameter (how much of the total thermal capacity sits in the fast-responding air node) across a deliberately generous range.


In [8]:
from thermal_counterfactual_gb.physics import two_node_conductances_w_per_k, simulate_2r2c_coastdown_hours

# f_air is DELIBERATE and swept, not independently grounded for this archetype
# (PROJECT.md Section 2.7, Ambiguity Is Informative). Air alone (no furnishings)
# is roughly 0.33 Wh/m3K x 175 m3 = ~0.06 kWh/K, under 1% of the total C=10
# kWh/K -- so even the LOW end of this sweep (2%) is already a generous
# allowance for light contents on top of bare air, and 20% is included only
# as a deliberately extreme upper bound for context, not a plausible estimate.
f_air_sweep = [0.02, 0.05, 0.10, 0.20]

print(f"{'state':>14} {'1R1C(h)':>9}", "".join(f"{'2R2C f=' + str(f):>12}" for f in f_air_sweep))
two_r_two_c_results = {}
for name, state in states.items():
    h_direct, h_im, h_mo = two_node_conductances_w_per_k(
        state,
        wall_area_m2=geometry["opaque_wall_area_m2"],
        roof_area_m2=geometry["roof_area_m2"],
        floor_area_m2=geometry["ground_floor_area_m2"],
        window_area_m2=geometry["window_area_m2"],
        heated_volume_m3=geometry["heated_volume_m3"],
    )
    row_values = []
    for f_air in f_air_sweep:
        t_hit = simulate_2r2c_coastdown_hours(
            f_air, c_kwh_per_k, h_direct, h_im, h_mo,
            t_start_c=comfort["preheat_ceiling_c"], t_min_c=comfort["minimum_c"], t_outdoor_c=t_outdoor_c,
            internal_gains_w=internal_gains_w,
        )
        row_values.append(t_hit)
    two_r_two_c_results[name] = row_values
    cd_1r1c = results[name]["coastdown_hours"]
    row_text = f"{name:>14} {cd_1r1c:>9.2f}"
    for t_hit in row_values:
        cell_text = f"{t_hit:>12.2f}" if t_hit is not None else f"{'>40':>12}"
        row_text += cell_text
    print(row_text)


         state   1R1C(h)  2R2C f=0.02 2R2C f=0.05  2R2C f=0.1  2R2C f=0.2


      baseline      5.00        1.64        2.09        2.79        3.85
      swi_only      5.98        2.04        2.49        3.23        4.41


 epc_c_package     18.67       11.42       11.93       12.76       14.27


In [9]:
peak_window_hours = 4  # 16:00-20:00, matching cold_snap_event.peak_window in config

baseline_2r2c = two_r_two_c_results["baseline"]
swi_2r2c = two_r_two_c_results["swi_only"]
epc_c_2r2c = two_r_two_c_results["epc_c_package"]

baseline_clears_peak = all(t is not None and t > peak_window_hours for t in baseline_2r2c)
swi_clears_peak = all(t is not None and t > peak_window_hours for t in swi_2r2c)
epc_c_clears_peak = all(t is not None and t > peak_window_hours for t in epc_c_2r2c)

print(f"Baseline clears the {peak_window_hours}h peak window across the full f_air sweep: {baseline_clears_peak}")
print(f"SWI-only clears the {peak_window_hours}h peak window across the full f_air sweep: {swi_clears_peak}")
print(f"EPC-C clears the {peak_window_hours}h peak window across the full f_air sweep: {epc_c_clears_peak}")
print()
swi_clears_only_at_max = (swi_2r2c[-1] is not None and swi_2r2c[-1] > peak_window_hours
                           and not all(t is not None and t > peak_window_hours for t in swi_2r2c[:-1]))

print(
    "This is a genuine, quantified finding from external review, not a footnote: the 1R1C model's "
    f"{results['baseline']['coastdown_hours']:.1f}h baseline coastdown is an UPPER BOUND, not a robust "
    "point estimate. A 2-node model, with the air-to-mass coupling grounded in BS EN ISO 6946's "
    "standard internal surface resistances (and now including the internal_gains_w netting from "
    "Section 5/6), shows the AIR node (what a thermostat and an occupant actually feel) cooling "
    "substantially faster than the single-node average temperature implies. Across a deliberately "
    f"generous f_air range (2-20% of total capacity assigned to the fast-responding air node), "
    "baseline fabric NEVER clears the 4-hour peak window, even with internal gains applied "
    f"(range {min(t for t in baseline_2r2c if t is not None):.2f}-{max(t for t in baseline_2r2c if t is not None):.2f}h, "
    "all below 4h). SWI-only is a narrower call than before gains were added: it now clears the window "
    f"ONLY at the single most generous point in the sweep (f_air=0.20, {swi_2r2c[-1]:.2f}h) while still "
    f"failing at every other point tested ({swi_2r2c[0]:.2f}-{swi_2r2c[-2]:.2f}h at f_air=0.02-0.10) -- "
    "a genuine change from the pre-gains result (where SWI-only failed everywhere in the sweep), but "
    "still not something to call reliable."
)
print()
print(
    f"EPC-C is the clearest story, and gains make it MORE robust, not less: its 2R2C coastdown range "
    f"is now {min(t for t in epc_c_2r2c if t is not None):.1f}-{max(t for t in epc_c_2r2c if t is not None):.1f}h "
    "across the same sweep (up from a pre-gains range of roughly 6.7-9.8h), comfortably above the "
    "4-hour peak window at every point. This is the internal-gains mechanism working exactly as "
    "expected: a fixed 400W gain is a small fraction of baseline's large loss and a much larger "
    "fraction of EPC-C's small one, so it helps the already-good state more than the already-bad one. "
    "The retrofit's safety margin is ROBUST to both the structural (1R1C-vs-2R2C) and the gains "
    "question; the baseline's apparent margin was not, and remains not, even once gains are added."
)
print()
print(
    "The steady-state kW reduction headline in Section 5 above is UNCHANGED in its DEPENDENCE on this "
    "structural question -- it still depends only on H, COP and now internal_gains_w, not on how C is "
    "split between air and mass, so it carries no structural-model risk of this kind."
)


Baseline clears the 4h peak window across the full f_air sweep: False
SWI-only clears the 4h peak window across the full f_air sweep: False
EPC-C clears the 4h peak window across the full f_air sweep: True

This is a genuine, quantified finding from external review, not a footnote: the 1R1C model's 5.0h baseline coastdown is an UPPER BOUND, not a robust point estimate. A 2-node model, with the air-to-mass coupling grounded in BS EN ISO 6946's standard internal surface resistances (and now including the internal_gains_w netting from Section 5/6), shows the AIR node (what a thermostat and an occupant actually feel) cooling substantially faster than the single-node average temperature implies. Across a deliberately generous f_air range (2-20% of total capacity assigned to the fast-responding air node), baseline fabric NEVER clears the 4-hour peak window, even with internal gains applied (range 1.64-3.85h, all below 4h). SWI-only is a narrower call than before gains were added: it now clea

## 6. Save to `data/intermediate/`\n\nParquet Handoff Rule (PROJECT.md Section 4.2): downstream notebooks read this file, never re-derive it.

In [10]:
import polars as pl

df = pl.DataFrame(
    [{"envelope_state": name, **r} for name, r in results.items()]
).select(
    pl.col("envelope_state").cast(pl.Utf8),
    pl.col("h_w_per_k").cast(pl.Float64),
    pl.col("tau_hours").cast(pl.Float64),
    pl.col("coastdown_hours").cast(pl.Float64),
    pl.col("peak_kw").cast(pl.Float64),
)

out_path = Path("../data/intermediate/01_archetype_physics.parquet")
out_path.parent.mkdir(parents=True, exist_ok=True)
df.write_parquet(out_path)
print(f"Wrote {out_path} ({df.height} rows)")
df


Wrote ../data/intermediate/01_archetype_physics.parquet (3 rows)


envelope_state,h_w_per_k,tau_hours,coastdown_hours,peak_kw
str,f64,f64,f64,f64
"""baseline""",272.75,36.663611,4.999726,2.4584
"""swi_only""",230.75,43.336945,5.982455,2.0552
"""epc_c_package""",85.55,116.890707,18.668201,0.66128
